In [1]:
"""
01_entrenar_modelo_final.py
Entrena y guarda el modelo final (ensamble LightGBM + XGBoost + CatBoost + meta-modelo logístico).

Entrada : train_data.jsonlines, test_data.jsonlines, y los 3 CSV de BERT (bert_feats_train.csv / bert_feats_test.csv)
Salida  : carpeta MODEL_DIR con los modelos entrenados (15 LGB + 15 XGB + 15 CatBoost + meta-modelo)
          y las matrices de features de test que esos modelos necesitan.
Configuración final: F_TEXT_OOF, F_KNN_TITLE, F_TIME_NEIGHBORS, F_WITHIN_SELLER_TITLE, F_SELLER_DEV, F_PAIR_TE,
                     3 BERT como feature, LGB 6 semillas, XGB 3, CatBoost 3, mezcla 'meta'.
"""
import os, re, unicodedata, warnings
import numpy as np, pandas as pd
from scipy import sparse
import joblib
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import NearestNeighbors
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# ============================ CONFIGURACIÓN ============================
TARGET = 'condition'; RANDOM_STATE = 42; N_FOLDS = 5
DATA_DIR = None                       # carpeta con train_data.jsonlines y test_data.jsonlines (None = buscar en /kaggle/input)
MODEL_DIR = 'modelo_final'
BERT_DIRS = ['/kaggle/input/notebooks/mirandacontieri/notebook-xlm-roberta-base',
             '/kaggle/input/notebooks/mirandacontieri/notebook-roberta-base-bne-v5-ap-2026',
             '/kaggle/input/notebooks/mirandacontieri/notebook-bert-titulos-ap']
LGB_LR = 0.03
LGB_SEEDS = [1, 2, 3, 4, 5, 6]
XGB_SEEDS = [RANDOM_STATE, 7, 11]
CAT_SEEDS = [RANDOM_STATE, 7, 11]
LGB_FIXED = dict(n_estimators=5000, subsample_freq=1, verbose=-1, n_jobs=-1)
LGB_BASE = dict(num_leaves=63, subsample=0.8, colsample_bytree=0.5, reg_lambda=1.0, min_child_samples=20)
os.makedirs(MODEL_DIR, exist_ok=True)

# ============================ DATOS ============================
dirs = [DATA_DIR] if DATA_DIR else ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub/competitions')]
paths = [os.path.join(r, f) for d in dirs if d and os.path.isdir(d) for r, _, fs in os.walk(d) for f in fs if f.endswith('.jsonlines')]
if not paths:
    import kagglehub
    base = kagglehub.competition_download('ap-2026-q-2')
    paths = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs if f.endswith('.jsonlines')]
pick = lambda w: [p for p in paths if w in os.path.basename(p).lower()][0]
train = pd.read_json(pick('train'), lines=True).reset_index(drop=True)
test = pd.read_json(pick('test'), lines=True).reset_index(drop=True)
ntr, nte = len(train), len(test)
classes = sorted(train[TARGET].unique())
yv = pd.Series((train[TARGET] == classes[-1]).astype(int).values)         # 'used' = 1
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)    # mismos folds en todo el script

# ============================ APLANADO Y LIMPIEZA ============================
def g(d, *keys):
    for k in keys:
        if isinstance(d, dict): d = d.get(k)
        else: return None
    return d
def to_dt(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_datetime(s, unit='ms', utc=True, errors='coerce')
    return pd.to_datetime(s, utc=True, errors='coerce')
def to_seconds(s):
    return (to_dt(s) - pd.Timestamp(0, tz='UTC')).dt.total_seconds().values

def flatten(df):
    o = pd.DataFrame(index=df.index)
    for c in ['price','base_price','initial_quantity','available_quantity','sold_quantity','seller_id',
              'listing_type_id','buying_mode','currency_id','status','category_id','accepts_mercadopago','automatic_relist']:
        o[c] = df[c]
    o['original_price_isna'] = df['original_price'].isna().astype(int)
    o['official_store'] = df['official_store_id'].notna().astype(int)
    o['has_video'] = df['video_id'].notna().astype(int)
    o['has_catalog'] = df['catalog_product_id'].notna().astype(int)
    o['has_parent'] = df['parent_item_id'].notna().astype(int)
    o['has_subtitle'] = df['subtitle'].notna().astype(int)
    o['state'] = df['seller_address'].apply(lambda d: g(d,'state','name'))
    o['city']  = df['seller_address'].apply(lambda d: g(d,'city','name'))
    o['free_shipping'] = df['shipping'].apply(lambda d: g(d,'free_shipping')).astype(float)
    o['local_pick_up'] = df['shipping'].apply(lambda d: g(d,'local_pick_up')).astype(float)
    o['shipping_mode'] = df['shipping'].apply(lambda d: g(d,'mode'))
    o['n_ship_methods'] = df['shipping'].apply(lambda d: len(g(d,'methods') or []))
    for name, col in [('n_pictures','pictures'), ('n_attributes','attributes'), ('n_variations','variations'), ('n_tags','tags'),
                      ('n_payment_methods','non_mercado_pago_payment_methods'), ('n_sub_status','sub_status')]:
        o[name] = df[col].apply(lambda x: len(x) if isinstance(x, list) else 0)
    tags_exp = df['tags'].apply(lambda x: x if isinstance(x, list) else [])
    for t in ['dragged_bids_and_visits','good_quality_thumbnail','poor_quality_thumbnail','free_relist','dragged_visits']:
        o['tag_'+t] = tags_exp.apply(lambda l: int(t in l))
    o['title'] = df['title'].fillna('')
    o['title_len'] = o['title'].str.len(); o['title_words'] = o['title'].str.split().str.len()
    o['title_upper_ratio'] = o['title'].apply(lambda s: sum(ch.isupper() for ch in s)/max(len(s),1))
    o['warranty'] = df['warranty'].fillna('').astype(str).str.lower().str.strip()
    dc, lu, st, et = to_dt(df['date_created']), to_dt(df['last_updated']), to_dt(df['start_time']), to_dt(df['stop_time'])
    o['created_year'] = dc.dt.year; o['created_month'] = dc.dt.month; o['created_dow'] = dc.dt.dayofweek
    o['days_active'] = (et - st).dt.total_seconds()/86400
    o['days_since_update'] = (lu - dc).dt.total_seconds()/86400
    o['attr_names'] = df['attributes'].apply(lambda l: ' | '.join(sorted({a.get('name','') for a in l})) if isinstance(l, list) else '')
    if TARGET in df.columns: o[TARGET] = df[TARGET]
    return o

def clean_missing(d, medians=None):
    d = d.copy()
    d['warranty_missing'] = (d['warranty'] == '').astype(int)
    d['warranty_has_no']  = d['warranty'].str.contains('sin garant', na=False).astype(int)
    d['warranty_yes']     = d['warranty'].str.contains(r'^(si|sí)$|meses|año|a\u00f1o|fabrica|f\u00e1brica|oficial', regex=True, na=False).astype(int)
    skip = {'title', 'attr_names', 'warranty', TARGET}
    cat = [c for c in d.columns if c not in skip and not pd.api.types.is_numeric_dtype(d[c])]
    d[cat] = d[cat].fillna('missing')
    num = d.select_dtypes(include=np.number).columns.difference([TARGET])
    if medians is None: medians = d[num].median()
    for c in num:
        if d[c].isna().any(): d[c+'_isna'] = d[c].isna().astype(int)
    d[num] = d[num].fillna(medians)
    return d, medians

LOG_COLS = ['price','base_price','initial_quantity','available_quantity','sold_quantity']
def treat_outliers(d, bounds=None):
    d = d.copy()
    for c in LOG_COLS: d[c+'_log'] = np.log1p(d[c].clip(lower=0))
    clip_cols = [c+'_log' for c in LOG_COLS] + ['title_len','n_pictures','n_attributes','days_active','days_since_update']
    if bounds is None: bounds = {c: (d[c].quantile(.005), d[c].quantile(.995)) for c in clip_cols}
    for c,(lo,hi) in bounds.items(): d[c] = d[c].clip(lo,hi)
    return d, bounds

df_c, MED = clean_missing(flatten(train)); df_o, BOUNDS = treat_outliers(df_c)
te_c, _   = clean_missing(flatten(test), medians=MED); te_o, _ = treat_outliers(te_c, bounds=BOUNDS)
for d in (df_o, te_o): d['seller_cat'] = d['seller_id'].astype(str) + '_' + d['category_id'].astype(str)

# ============================ FEATURES BASE ============================
NUM = ['price_log','base_price_log','initial_quantity_log','available_quantity_log','sold_quantity_log',
       'title_len','title_words','title_upper_ratio','n_pictures','n_attributes','n_variations','n_tags','n_payment_methods',
       'n_ship_methods','days_active','days_since_update','created_year','created_month','created_dow',
       'free_shipping','local_pick_up','accepts_mercadopago','automatic_relist','official_store','has_video','has_catalog',
       'has_parent','has_subtitle','original_price_isna','warranty_missing','warranty_has_no','warranty_yes'] + \
      [c for c in df_o.columns if c.startswith('tag_')]
NUM = [c for c in NUM if c in df_o.columns]
LOWCAT = ['listing_type_id','buying_mode','status','currency_id','shipping_mode','state']
HIGHCAT = ['category_id','seller_id','city']

def freq_enc(train_s, s):
    return s.map(train_s.value_counts(normalize=True)).fillna(0)
def base_X(d):
    Z = d[NUM + LOWCAT].copy().reset_index(drop=True)
    for c in HIGHCAT: Z[c+'_freq'] = freq_enc(df_o[c], d[c]).reset_index(drop=True)
    return pd.get_dummies(Z, columns=LOWCAT)
Xa = base_X(df_o); Xb = base_X(te_o).reindex(columns=Xa.columns, fill_value=0)

allc = pd.concat([df_o[['seller_id','price_log']], te_o[['seller_id','price_log']]])
agg = allc.groupby('seller_id')['price_log'].agg(['count','mean','std'])
for d, Z in [(df_o, Xa), (te_o, Xb)]:
    for s in ['count','mean','std']: Z['seller_'+s] = d['seller_id'].map(agg[s]).fillna(0).values

def target_encode(col_tr, col_te, smooth=10):
    prior = yv.mean(); oof = np.zeros(len(col_tr)); col_tr = col_tr.reset_index(drop=True)
    for tr_i, va_i in skf.split(np.zeros(len(col_tr)), yv):
        s = pd.DataFrame({'c': col_tr.iloc[tr_i].values, 'y': yv.iloc[tr_i].values}).groupby('c')['y'].agg(['sum','count'])
        enc = (s['sum'] + prior*smooth) / (s['count'] + smooth)
        oof[va_i] = col_tr.iloc[va_i].map(enc).fillna(prior).values
    s = pd.DataFrame({'c': col_tr.values, 'y': yv.values}).groupby('c')['y'].agg(['sum','count'])
    enc = (s['sum'] + prior*smooth) / (s['count'] + smooth)
    return oof, col_te.reset_index(drop=True).map(enc).fillna(prior).values
for c in ['seller_id', 'category_id', 'city', 'seller_cat']:
    Xa[c+'_te'], Xb[c+'_te'] = target_encode(df_o[c], te_o[c])

cols = ['category_id', 'seller_id', 'price_log']
comb = pd.concat([df_o[cols], te_o[cols]], ignore_index=True)
g_cat = comb.groupby('category_id')['price_log']
comb['price_rel_cat']    = comb['price_log'] - g_cat.transform('median')
comb['price_z_cat']      = comb['price_rel_cat'] / (g_cat.transform('std').fillna(0) + 1e-6)
comb['price_rank_cat']   = g_cat.rank(pct=True)
comb['cat_n']            = g_cat.transform('count')
comb['price_rel_seller'] = comb['price_log'] - comb.groupby('seller_id')['price_log'].transform('mean')
for c in ['price_rel_cat', 'price_z_cat', 'price_rank_cat', 'cat_n', 'price_rel_seller']:
    Xa[c] = comb[c].values[:ntr]; Xb[c] = comb[c].values[ntr:]
Xa['sold_frac'] = (df_o['sold_quantity'] / (df_o['initial_quantity'] + 1)).values
Xb['sold_frac'] = (te_o['sold_quantity'] / (te_o['initial_quantity'] + 1)).values

# ============================ FEATURES DE TEXTO ============================
def _norm(s):
    s = unicodedata.normalize('NFKD', str(s).lower()); return ''.join(ch for ch in s if not unicodedata.combining(ch))
def _tok(x): return re.sub(r'[^a-z0-9]+', '_', _norm(x)).strip('_')
L = lambda v: v if isinstance(v, list) else []

def build_sources(raw):
    raw = raw.reset_index(drop=True); S = pd.DataFrame(index=raw.index)
    S['title']    = raw['title'].fillna('').map(_norm)
    S['warranty'] = raw['warranty'].fillna('sin_dato').map(_norm)
    S['attr_ids'] = raw['attributes'].map(lambda v: ' '.join(_tok(a.get('id', '')) for a in L(v)))
    S['attr_vals'] = raw['attributes'].map(lambda v: ' '.join(f"{_tok(a.get('id',''))}__{_tok(a.get('value_name',''))}" for a in L(v)))
    S['variations'] = raw['variations'].map(lambda v: ' '.join(f"{_tok(c.get('name',''))}__{_tok(c.get('value_name',''))}"
                                                              for x in L(v) for c in L(x.get('attribute_combinations'))))
    S['payments'] = raw['non_mercado_pago_payment_methods'].map(lambda v: ' '.join(_tok(p.get('description', '')) for p in L(v)))
    S['pics'] = raw['pictures'].map(lambda v: ' '.join(f"sz_{p.get('size','')} mx_{p.get('max_size','')}" for p in L(v)))
    dom = raw['permalink'].fillna('').str.extract(r'https?://([^./]+)\.')[0].fillna('none')
    S['meta'] = (raw['tags'].map(lambda v: ' '.join(_tok(t) for t in L(v))) + ' dom_' + dom
                 + ' ' + raw['sub_status'].map(lambda v: ' '.join('ss_' + _tok(t) for t in L(v))))
    return S
Sa, Sb = build_sources(train), build_sources(test)

def text_numeric_feats(raw):
    raw = raw.reset_index(drop=True); t = raw['title'].fillna(''); F = pd.DataFrame(index=raw.index)
    F['t_digits'] = t.str.count(r'\d'); F['t_digit_ratio'] = F['t_digits'] / (t.str.len() + 1)
    F['t_has_year'] = t.str.contains(r'\b(19|20)\d{2}\b').astype(int)
    F['t_model_code'] = t.str.count(r'\b(?=\w*\d)(?=\w*[A-Za-z])\w{3,}\b')
    F['t_caps_words'] = t.str.count(r'\b[A-Z]{3,}\b')
    F['t_all_lower'] = (t == t.str.lower()).astype(int); F['t_title_case'] = t.str.istitle().astype(int)
    F['t_punct'] = t.str.count(r'[!¡?\*\-/()\.,]')
    F['t_has_money'] = t.str.contains(r'\$|usd|u\$s', case=False).astype(int)
    F['t_has_qty'] = t.str.contains(r'\b(x\s?\d+|\d+\s?(un|u|unid|unidades|pcs|piezas))\b', case=False).astype(int)
    F['t_has_unit'] = t.str.contains(r'\b\d+\s?(gb|tb|mb|mm|cm|kg|gr|lts?|ml|w|hz|mah|ghz|pulgadas|")\b', case=False).astype(int)
    F['t_longest_tok'] = t.str.split().map(lambda w: max([len(x) for x in w], default=0))
    F['t_dup_words'] = t.str.split().map(lambda w: len(w) - len({x.lower() for x in w}))
    P = raw['pictures'].map(lambda v: v if isinstance(v, list) else [])
    def wh(p, key):
        m = re.match(r'(\d+)x(\d+)', p.get(key, '') or ''); return (int(m[1]), int(m[2])) if m else (np.nan, np.nan)
    first = P.map(lambda v: wh(v[0], 'size') if v else (np.nan, np.nan))
    firstmx = P.map(lambda v: wh(v[0], 'max_size') if v else (np.nan, np.nan))
    F['pic_w'] = first.map(lambda x: x[0]); F['pic_h'] = first.map(lambda x: x[1])
    F['pic_mw'] = firstmx.map(lambda x: x[0]); F['pic_mh'] = firstmx.map(lambda x: x[1])
    F['pic_ratio'] = F['pic_w'] / F['pic_h']; F['pic_max_mpx'] = F['pic_mw'] * F['pic_mh'] / 1e6
    F['pic_n_sizes'] = P.map(lambda v: len({p.get('size') for p in v}))
    F['pic_own_id'] = [np.mean([i in p.get('id', '') for p in v]) if v else np.nan for i, v in zip(raw['id'], P)]
    pdate = P.map(lambda v: re.search(r'_(\d{2})(\d{4})$', v[0].get('id', '')) if v else None)
    py = pdate.map(lambda m: int(m[2]) if m else np.nan); pm = pdate.map(lambda m: int(m[1]) if m else np.nan)
    dc = to_dt(raw['date_created'])
    F['pic_year'] = py; F['pic_gap_months'] = (dc.dt.year * 12 + dc.dt.month) - (py * 12 + pm)
    F['n_attr_vals_empty'] = raw['attributes'].map(lambda v: sum(1 for a in v if not a.get('value_name')) if isinstance(v, list) else 0)
    return F

USED_KW = ['estado','muy buen','lote','revista','antigua','antiguo','vinilo','lp','ano','made in',
           'usado','usada','usados','usadas','segunda mano','impecable','poco uso','detalles','para repuestos']
NEW_KW  = ['nuevo','nueva','nuevos','nuevas','kit','original','envio gratis','libro digital','sellado','sellada','sin abrir','0km','0 km','en caja']
WARR_USED = ['estado','buen','mi reputacion','mis calificaciones','calificaciones','descripcion','sin garantia']
WARR_NEW  = ['meses','ano','anos','de garantia','fabrica']
kw_re = lambda kws: re.compile(r'\b(' + '|'.join(re.escape(k) for k in kws) + r')\b')
RE_U, RE_N, RE_WU, RE_WN = kw_re(USED_KW), kw_re(NEW_KW), kw_re(WARR_USED), kw_re(WARR_NEW)
def kw_counts(S):
    return pd.DataFrame({'tx_kw_used': S['title'].map(lambda s: len(RE_U.findall(s))).values,
                         'tx_kw_new': S['title'].map(lambda s: len(RE_N.findall(s))).values,
                         'tx_kw_used_warr': S['warranty'].map(lambda s: len(RE_WU.findall(s))).values,
                         'tx_kw_new_warr': S['warranty'].map(lambda s: len(RE_WN.findall(s))).values})

views = {
  'title_w':  (Sa['title'],    Sb['title'],    TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=80000, sublinear_tf=True)),
  'title_c':  (Sa['title'],    Sb['title'],    TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=120000, sublinear_tf=True)),
  'warranty': (Sa['warranty'], Sb['warranty'], TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
  'attrs':    (Sa['attr_vals'] + ' ' + Sa['attr_ids'], Sb['attr_vals'] + ' ' + Sb['attr_ids'], TfidfVectorizer(token_pattern=r'\S+', min_df=3, max_features=60000)),
  'vars':     (Sa['variations'], Sb['variations'], TfidfVectorizer(token_pattern=r'\S+', min_df=3)),
  'pics_meta':(Sa['pics'] + ' ' + Sa['meta'] + ' ' + Sa['payments'], Sb['pics'] + ' ' + Sb['meta'] + ' ' + Sb['payments'], TfidfVectorizer(token_pattern=r'\S+', min_df=3)),
}
mats = {n: (vec.fit_transform(a), vec.transform(b)) for n, (a, b, vec) in views.items()}
Mtr_all = sparse.hstack([m[0] for m in mats.values()]).tocsr(); Mte_all = sparse.hstack([m[1] for m in mats.values()]).tocsr()

# ============================ BLOQUES DE FEATURES ============================
def block_text_basic():
    Fa = text_numeric_feats(train).add_prefix('tx_'); Fb = text_numeric_feats(test).add_prefix('tx_')
    ka, kb = kw_counts(Sa), kw_counts(Sb)
    for c in ka.columns: Fa[c] = ka[c].values; Fb[c] = kb[c].values
    return Fa, Fb

def block_text_oof():
    Fa, Fb = {}, {}
    def add(name, Mtr, Mte, model):
        Fa['tx_oof_' + name] = cross_val_predict(model, Mtr, yv, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
        Fb['tx_oof_' + name] = model.fit(Mtr, yv).predict_proba(Mte)[:, 1]
    for name, (Mtr, Mte) in mats.items(): add(name, Mtr, Mte, LogisticRegression(max_iter=3000, C=3))
    add('all', Mtr_all, Mte_all, LogisticRegression(max_iter=3000, C=1))
    cvb = CountVectorizer(ngram_range=(1, 2), min_df=2, binary=True)
    add('nb_title', cvb.fit_transform(Sa['title']), cvb.transform(Sb['title']), ComplementNB(alpha=0.5))
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

def block_knn_title():
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Mtr = vec.fit_transform(df_o['title']); Mte = vec.transform(te_o['title']); K = 10
    nn = NearestNeighbors(n_neighbors=K + 1, metric='cosine', algorithm='brute', n_jobs=-1).fit(Mtr)
    def knn_feats(M, is_train):
        out = []
        for s in range(0, M.shape[0], 5000):
            d, ix = nn.kneighbors(M[s:s + 5000])
            if is_train: d = np.where(ix == np.arange(s, s + ix.shape[0])[:, None], np.inf, d)
            o = np.argsort(d, axis=1)[:, :K]
            d = np.take_along_axis(d, o, 1); ix = np.take_along_axis(ix, o, 1)
            sim = 1 - d; lab = yv.values[ix]; w = sim + 1e-6
            out.append(np.c_[(lab * w).sum(1) / w.sum(1), lab[:, :3].mean(1), lab[:, 0], sim[:, 0], sim.mean(1)])
        return np.vstack(out)
    names = ['knn_wmean', 'knn_top3', 'knn_top1', 'knn_sim1', 'knn_simmean']
    return pd.DataFrame(knn_feats(Mtr, True), columns=names), pd.DataFrame(knn_feats(Mte, False), columns=names)

def _fold_id():
    fid = np.zeros(ntr, int)
    for k, (_, va_i) in enumerate(skf.split(np.zeros(ntr), yv)): fid[va_i] = k
    return fid
def _groups():
    s_tr, s_te = train['seller_id'].values, test['seller_id'].values
    return (pd.Series(np.arange(ntr)).groupby(s_tr).apply(np.array), pd.Series(np.arange(nte)).groupby(s_te).apply(np.array))

def block_time_neighbors():
    ts_tr, ts_te = to_seconds(train['date_created']), to_seconds(test['date_created'])
    Y = yv.values.astype(float); fold_id = _fold_id(); groups_tr, groups_te = _groups()
    tcols = ['ti_n', 'ti_nn_lab', 'ti_nn_gap', 'ti_k3', 'ti_w60', 'ti_w3600', 'ti_prev_lab', 'ti_prev_gap',
             'ti_next_lab', 'ti_next_gap', 'ti_n60', 'ti_pos60', 'ti_n3600', 'ti_rate3600']
    def feats_time(tq, trf, valid, y_ref):
        q = len(tq); out = np.full((q, len(tcols)), np.nan); ar = np.arange(q)
        D = np.abs(tq[:, None] - trf[None, :]); D2 = np.where(valid, D, np.inf)
        n = valid.sum(1); has = n > 0; out[:, 0] = n
        j = D2.argmin(1); out[has, 1] = y_ref[j][has]; out[has, 2] = np.log1p(D2[ar, j])[has]
        o3 = np.argsort(D2, axis=1)[:, :3]; ok3 = np.isfinite(D2[ar[:, None], o3])
        out[has, 3] = ((y_ref[o3] * ok3).sum(1) / np.maximum(ok3.sum(1), 1))[has]
        for col, tau in [(4, 60.0), (5, 3600.0)]:
            w = np.where(valid, np.exp(-D / tau), 0.0); sw = w.sum(1)
            out[:, col] = np.where(sw > 1e-9, (w * y_ref).sum(1) / np.maximum(sw, 1e-12), np.nan)
        gap_prev = np.where(valid & (trf[None, :] <= tq[:, None]), tq[:, None] - trf[None, :], np.inf)
        gap_next = np.where(valid & (trf[None, :] >= tq[:, None]), trf[None, :] - tq[:, None], np.inf)
        for (a, b), G in [((6, 7), gap_prev), ((8, 9), gap_next)]:
            jj = G.argmin(1); ok = np.isfinite(G[ar, jj])
            out[ok, a] = y_ref[jj][ok]; out[ok, b] = np.log1p(G[ar, jj])[ok]
        m60 = valid & (D <= 60); m1h = valid & (D <= 3600)
        out[:, 10] = m60.sum(1); out[:, 11] = (m60 * y_ref).sum(1); out[:, 12] = m1h.sum(1)
        n1h = m1h.sum(1); out[n1h > 0, 13] = ((m1h * y_ref).sum(1) / np.maximum(n1h, 1))[n1h > 0]
        return out
    T_tr = np.full((ntr, len(tcols)), np.nan); T_te = np.full((nte, len(tcols)), np.nan)
    for s, idx in groups_tr.items():
        T_tr[idx] = feats_time(ts_tr[idx], ts_tr[idx], fold_id[idx][:, None] != fold_id[idx][None, :], Y[idx])
    for s, idx in groups_te.items():
        ref = groups_tr.get(s)
        if ref is None: T_te[idx, 0] = 0; T_te[idx, 10] = 0; T_te[idx, 11] = 0; T_te[idx, 12] = 0; continue
        T_te[idx] = feats_time(ts_te[idx], ts_tr[ref], np.ones((len(idx), len(ref)), bool), Y[ref])
    return pd.DataFrame(T_tr, columns=tcols), pd.DataFrame(T_te, columns=tcols)

def block_within_seller_title():
    Mtr_t, Mte_t = mats['title_w']
    lp_tr = np.log1p(train['price'].clip(lower=0).values); lp_te = np.log1p(test['price'].clip(lower=0).values)
    Y = yv.values.astype(float); fold_id = _fold_id(); groups_tr, groups_te = _groups()
    wcols = ['wi_n', 'wi_pos', 'wi_posrate', 'wi_top1_lab', 'wi_top1_sim', 'wi_wmean', 'wi_price_lab', 'wi_price_gap']
    def feats(S, valid, y_ref, dprice):
        q = S.shape[0]; out = np.full((q, len(wcols)), np.nan); ar = np.arange(q)
        n = valid.sum(1); has = n > 0
        S2 = np.where(valid, S, -np.inf); j = S2.argmax(1); D2 = np.where(valid, dprice, np.inf); jp = D2.argmin(1)
        w = np.where(valid, np.exp(10 * S), 0.0); pos = (valid * y_ref).sum(1)
        out[:, 0] = n; out[:, 1] = pos; out[has, 2] = (pos / np.maximum(n, 1))[has]
        out[has, 3] = y_ref[j][has]; out[has, 4] = S2[ar, j][has]
        out[has, 5] = ((w * y_ref).sum(1) / np.maximum(w.sum(1), 1e-12))[has]
        out[has, 6] = y_ref[jp][has]; out[has, 7] = D2[ar, jp][has]
        return out
    W_tr = np.full((ntr, len(wcols)), np.nan); W_te = np.full((nte, len(wcols)), np.nan)
    for s, idx in groups_tr.items():
        S = (Mtr_t[idx] @ Mtr_t[idx].T).toarray()
        W_tr[idx] = feats(S, fold_id[idx][:, None] != fold_id[idx][None, :], Y[idx], np.abs(lp_tr[idx][:, None] - lp_tr[idx][None, :]))
    for s, idx in groups_te.items():
        ref = groups_tr.get(s)
        if ref is None: W_te[idx, 0] = 0; W_te[idx, 1] = 0; continue
        S = (Mte_t[idx] @ Mtr_t[ref].T).toarray()
        W_te[idx] = feats(S, np.ones(S.shape, bool), Y[ref], np.abs(lp_te[idx][:, None] - lp_tr[ref][None, :]))
    return pd.DataFrame(W_tr, columns=wcols), pd.DataFrame(W_te, columns=wcols)

def _pic1(raw):
    return raw['pictures'].map(lambda l: (l[0].get('size') or 'none') if isinstance(l, list) and l else 'none')
def _warr_bucket(S):
    w = S['warranty']
    return np.select([w.str.contains('sin garantia').values, w.str.contains(r'\d+\s*(mes|ano)').values, (w == 'sin_dato').values],
                     ['sin', 'dur', 'none'], 'otro')

def block_seller_dev():
    def table(d, raw, S):
        wh = np.array([(int(m[1]), int(m[2])) if m else (np.nan, np.nan)
                       for m in [re.match(r'(\d+)x(\d+)', sz) for sz in _pic1(raw).values]], dtype=float)
        return pd.DataFrame({'seller': d['seller_id'].astype(str).values,
            'price_log': d['price_log'].values, 'n_pictures': d['n_pictures'].values, 'title_len': d['title_len'].values,
            'n_attributes': d['n_attributes'].values, 'initial_quantity_log': d['initial_quantity_log'].values,
            'sold_quantity_log': d['sold_quantity_log'].values, 'pic_ratio': wh[:, 0] / wh[:, 1],
            'pic_size': _pic1(raw).values, 'warr': _warr_bucket(S), 'listing_type': d['listing_type_id'].astype(str).values,
            'ship_mode': d['shipping_mode'].astype(str).values, 'free_shipping': d['free_shipping'].astype(str).values,
            'category': d['category_id'].astype(str).values})
    comb = pd.concat([table(df_o, train, Sa), table(te_o, test, Sb)], ignore_index=True)
    sel = comb['seller']; out = pd.DataFrame(index=comb.index)
    for c in ['price_log', 'n_pictures', 'title_len', 'n_attributes', 'initial_quantity_log', 'sold_quantity_log', 'pic_ratio']:
        x = comb[c].astype(float); has = x.notna(); xs = x.where(has, 0.0)
        cnt = has.astype(int).groupby(sel).transform('sum'); sm = xs.groupby(sel).transform('sum')
        loo_n = (cnt - has.astype(int)).values
        loo_mean = np.where(loo_n > 0, (sm - xs).values / np.maximum(loo_n, 1), np.nan)
        out['sd_dev_' + c] = np.where(has.values & (loo_n > 0), x.values - loo_mean, np.nan)
    n_sel = sel.map(sel.value_counts()).values
    for c in ['pic_size', 'warr', 'listing_type', 'ship_mode', 'free_shipping', 'category']:
        key = sel + '|' + comb[c]; n_pair = key.map(key.value_counts()).values
        out['sd_same_' + c] = np.where(n_sel > 1, (n_pair - 1) / np.maximum(n_sel - 1, 1), np.nan)
        out['sd_nuniq_' + c] = comb.groupby('seller')[c].transform('nunique').values
    return out.iloc[:ntr].reset_index(drop=True), out.iloc[ntr:].reset_index(drop=True)

def block_pair_te():
    q = np.unique(np.quantile(Xa['price_rel_cat'], np.linspace(0, 1, 6)[1:-1]))
    def table(d, raw, S, rel):
        return pd.DataFrame({'seller_id': d['seller_id'].astype(str).values, 'category_id': d['category_id'].astype(str).values,
                             'pic_size': _pic1(raw).values, 'warr': _warr_bucket(S), 'listing_type_id': d['listing_type_id'].astype(str).values,
                             'free_shipping': d['free_shipping'].astype(str).values, 'price_relbin': np.digitize(rel, q).astype(str)})
    Ta = table(df_o, train, Sa, Xa['price_rel_cat'].values); Tb = table(te_o, test, Sb, Xb['price_rel_cat'].values)
    prior = float(yv.mean()); yarr = yv.values.astype(float)
    def hier_te(a_tr, b_tr, a_te, b_te, sm_a=10.0, sm_ab=5.0):
        ab_tr = a_tr + '|' + b_tr; ab_te = a_te + '|' + b_te
        def fit(idx):
            da = pd.DataFrame({'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('a')['y'].agg(['sum', 'count'])
            pa = (da['sum'] + prior * sm_a) / (da['count'] + sm_a)
            dab = pd.DataFrame({'ab': ab_tr.values[idx], 'a': a_tr.values[idx], 'y': yarr[idx]}).groupby('ab').agg(s=('y', 'sum'), n=('y', 'count'), a=('a', 'first'))
            dab['enc'] = (dab['s'] + dab['a'].map(pa).values * sm_ab) / (dab['n'] + sm_ab)
            return pa, dab
        def apply(pa, dab, a, ab):
            return ab.map(dab['enc']).fillna(a.map(pa).fillna(prior)).values, ab.map(dab['n']).fillna(0).values
        enc = np.zeros(ntr); cnt = np.zeros(ntr)
        for tr_i, va_i in skf.split(np.zeros(ntr), yv):
            pa, dab = fit(tr_i); enc[va_i], cnt[va_i] = apply(pa, dab, a_tr.iloc[va_i], ab_tr.iloc[va_i])
        pa, dab = fit(np.arange(ntr)); te_e, te_c = apply(pa, dab, a_te, ab_te)
        return enc, cnt, te_e, te_c
    short = {'seller_id': 'sel', 'category_id': 'cat', 'listing_type_id': 'lt'}
    PAIRS = [('seller_id', 'pic_size'), ('seller_id', 'price_relbin'), ('seller_id', 'warr'), ('seller_id', 'listing_type_id'),
             ('seller_id', 'free_shipping'), ('category_id', 'pic_size'), ('category_id', 'price_relbin'), ('category_id', 'warr'),
             ('category_id', 'listing_type_id'), ('category_id', 'free_shipping')]
    Fa, Fb = {}, {}
    for A, B in PAIRS:
        nm = f'{short.get(A, A)}_{short.get(B, B)}'
        e, c, te_e, te_c = hier_te(Ta[A], Ta[B], Tb[A], Tb[B])
        Fa['pte_' + nm], Fa['ptn_' + nm], Fb['pte_' + nm], Fb['ptn_' + nm] = e, c, te_e, te_c
    return pd.DataFrame(Fa), pd.DataFrame(Fb)

# el orden de los bloques define el orden de las columnas (importa para reproducir los modelos)
for fn in [block_text_basic, block_text_oof, block_knn_title, block_time_neighbors,
           block_within_seller_title, block_seller_dev, block_pair_te]:
    Fa, Fb = fn()
    Xa = pd.concat([Xa, Fa.reset_index(drop=True)], axis=1); Xb = pd.concat([Xb, Fb.reset_index(drop=True)], axis=1)

for i_, d_ in enumerate(BERT_DIRS):
    nm_ = 'bert' if i_ == 0 else f'bert{i_ + 1}'
    a_ = pd.read_csv(f'{d_}/bert_feats_train.csv'); b_ = pd.read_csv(f'{d_}/bert_feats_test.csv')
    assert len(a_) == ntr and len(b_) == nte and (a_['y_check'].values == yv.values).all(), f'{d_}: el BERT no coincide con este train'
    Xa = pd.concat([Xa, a_[['bert_oof']].rename(columns={'bert_oof': nm_ + '_oof'}).reset_index(drop=True)], axis=1)
    Xb = pd.concat([Xb, b_[['bert_oof']].rename(columns={'bert_oof': nm_ + '_oof'}).reset_index(drop=True)], axis=1)
Xa = Xa.copy(); Xb = Xb[Xa.columns].copy()
print('Matriz final:', Xa.shape, '(train) |', Xb.shape, '(test)')

# ============================ MODELOS ============================
folds = list(skf.split(np.zeros(ntr), yv))
preds_oof, models = {}, {}

# --- LightGBM ---
oofs, ms = [], []
for seed in LGB_SEEDS:
    params = {**LGB_FIXED, **LGB_BASE, 'learning_rate': LGB_LR, 'random_state': seed}
    oof = np.zeros(ntr)
    for tr_i, va_i in folds:
        m = lgb.LGBMClassifier(**params)
        m.fit(Xa.iloc[tr_i], yv.iloc[tr_i], eval_set=[(Xa.iloc[va_i], yv.iloc[va_i])], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va_i] = m.predict_proba(Xa.iloc[va_i])[:, 1]; ms.append(m)
    oofs.append(oof); print(f'LightGBM semilla {seed} lista')
preds_oof['lgb'] = np.mean(oofs, axis=0); models['lgb'] = ms

# --- XGBoost ---
clean_cols = lambda d: d.set_axis([re.sub(r'[\[\]<>]', '_', str(c)) for c in d.columns], axis=1)
Xa_x = clean_cols(Xa)
oofs, ms = [], []
for seed in XGB_SEEDS:
    oof = np.zeros(ntr)
    for tr_i, va_i in folds:
        m = xgb.XGBClassifier(n_estimators=5000, learning_rate=LGB_LR, max_depth=7, min_child_weight=2, subsample=0.8, colsample_bytree=0.5,
                              reg_lambda=1.0, tree_method='hist', eval_metric='logloss', early_stopping_rounds=100, n_jobs=-1, random_state=seed)
        m.fit(Xa_x.iloc[tr_i], yv.iloc[tr_i], eval_set=[(Xa_x.iloc[va_i], yv.iloc[va_i])], verbose=False)
        oof[va_i] = m.predict_proba(Xa_x.iloc[va_i])[:, 1]; ms.append(m)
    oofs.append(oof); print(f'XGBoost semilla {seed} lista')
preds_oof['xgb'] = np.mean(oofs, axis=0); models['xgb'] = ms

# --- CatBoost (no ve las features que resumen la etiqueta del vendedor; recibe vendedor/categoría/ciudad como categóricas) ---
is_label_feat = lambda c: c.endswith('_te') or c.startswith(('knn_', 'wi_', 'ti_', 'pte_', 'ptn_'))
num_cols = [c for c in Xa.columns if not is_label_feat(c)]
cat_frame = lambda raw, o: pd.DataFrame({'seller_id': raw['seller_id'].astype(str).values, 'category_id': raw['category_id'].astype(str).values,
                                         'city': o['city'].astype(str).values, 'seller_cat': o['seller_cat'].astype(str).values})
Xc_tr = pd.concat([Xa[num_cols].reset_index(drop=True), cat_frame(train, df_o)], axis=1)
Xc_te = pd.concat([Xb[num_cols].reset_index(drop=True), cat_frame(test, te_o)], axis=1)
cat_idx = list(range(len(num_cols), Xc_tr.shape[1])); Y = yv.values
oofs, ms = [], []
for seed in CAT_SEEDS:
    oof = np.zeros(ntr)
    for tr_i, va_i in folds:
        m = CatBoostClassifier(iterations=2000, learning_rate=0.08, depth=8, l2_leaf_reg=3, eval_metric='Logloss',
                               early_stopping_rounds=150, thread_count=-1, random_seed=seed, verbose=0)
        m.fit(Xc_tr.iloc[tr_i], Y[tr_i], cat_features=cat_idx, eval_set=(Xc_tr.iloc[va_i], Y[va_i]))
        oof[va_i] = m.predict_proba(Xc_tr.iloc[va_i])[:, 1]; ms.append(m)
    oofs.append(oof); print(f'CatBoost semilla {seed} lista')
preds_oof['cat'] = np.mean(oofs, axis=0); models['cat'] = ms

# --- Meta-modelo: regresión logística sobre los logits de las predicciones OOF ---
names = ['lgb', 'xgb', 'cat']
lg = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
O = np.c_[[preds_oof[n] for n in names]].T
meta = LogisticRegression(C=1.0).fit(lg(O), yv)
print('pesos del meta-modelo:', dict(zip(names, meta.coef_[0].round(2))))

# ============================ GUARDADO ============================
joblib.dump(models['lgb'], os.path.join(MODEL_DIR, 'lgb_models.joblib'))
joblib.dump(models['xgb'], os.path.join(MODEL_DIR, 'xgb_models.joblib'))
joblib.dump(models['cat'], os.path.join(MODEL_DIR, 'cat_models.joblib'))
joblib.dump({'meta': meta, 'names': names, 'classes': classes, 'n_folds': N_FOLDS,
             'n_seeds': {'lgb': len(LGB_SEEDS), 'xgb': len(XGB_SEEDS), 'cat': len(CAT_SEEDS)}}, os.path.join(MODEL_DIR, 'meta.joblib'))
pd.to_pickle(Xb, os.path.join(MODEL_DIR, 'X_test_lgb_xgb.pkl'))      # features de test para LightGBM y XGBoost
pd.to_pickle(Xc_te, os.path.join(MODEL_DIR, 'X_test_cat.pkl'))       # features de test para CatBoost
print('Modelo final guardado en', MODEL_DIR)

Matriz final: (70000, 203) (train) | (30000, 203) (test)
LightGBM semilla 1 lista
LightGBM semilla 2 lista
LightGBM semilla 3 lista
LightGBM semilla 4 lista
LightGBM semilla 5 lista
LightGBM semilla 6 lista
XGBoost semilla 42 lista
XGBoost semilla 7 lista
XGBoost semilla 11 lista
CatBoost semilla 42 lista
CatBoost semilla 7 lista
CatBoost semilla 11 lista
pesos del meta-modelo: {'lgb': np.float64(0.34), 'xgb': np.float64(0.38), 'cat': np.float64(0.25)}
Modelo final guardado en modelo_final
